# 01 — RAG Pipeline Basics: Walkthrough Toàn Bộ Pipeline

**Vai trò:** Pipeline Engineer · **Task:** S3-PE-07 (Yêu cầu 9.3)

Notebook này đưa bạn qua **6 bước** của RAG pipeline từ đầu đến cuối — từ tải tài liệu, chia nhỏ văn bản, tạo embedding, lưu vào vector store, xây dựng prompt, đến sinh câu trả lời. Đây là "bức tranh toàn cảnh" của Pipeline Engineer trước khi đào sâu vào từng thành phần ở các notebook tiếp theo.

```
Indexing : DocumentLoader → TextChunker → OllamaEmbeddingModel → ChromaVectorStore
Querying : embed(question) → similarity_search → PromptBuilder → OllamaClient → RAGResponse
```

**Trạng thái component khi chạy notebook này:**
- `DocumentLoader` — ✅ thật (S1-DE-01/02)
- `OllamaClient.is_available()` / `list_models()` — ✅ thật (S1-ME-01)
- `TextChunker`, `OllamaEmbeddingModel`, `ChromaVectorStore`, `PromptBuilder` — 🔶 stub; notebook dùng mock nội tuyến để chạy đủ luồng mà không lỗi
- `OllamaClient.generate()` — 🔶 stub (S3-ME-01); notebook gọi nếu server chạy, graceful-fallback nếu không

## 0. Setup — Thêm Project Root vào `sys.path`

In [1]:
import sys
import math
import hashlib
import time
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in (".git", "pyproject.toml", "CLAUDE.md")):
            return parent
    return start

PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import data models và interfaces dùng chung
from src.models import (
    Document, DocumentType,
    Chunk, ChunkStrategy,
    EmbeddingVector,
    ScoredChunk,
    RAGResponse,
    IndexingResult,
)
from src.interfaces import (
    BaseChunker,
    BaseEmbeddingModel,
    BaseVectorStore,
    BaseLLMClient,
)

# Import component đã triển khai thật
from src.data.loader import DocumentLoader
from src.generation.llm_client import OllamaClient
from src.pipeline.rag_pipeline import RAGPipeline
from config.settings import AppConfig

print(f"Project root : {PROJECT_ROOT}")
print("Import thành công.")


Project root : D:\lh222k\AI-Research-Assistant-with-RAG
Import thành công.


## 1. Kiểm Tra Môi Trường OLLAMA

`OllamaClient.is_available()` trả về `bool` và **không bao giờ ném exception** dù server tắt (Yêu cầu 5.2). Notebook an toàn chạy tiếp dù OLLAMA offline — các cell sinh câu trả lời sẽ tự fallback.

In [2]:
cfg = AppConfig.from_env()

llm_client = OllamaClient(
    model_name=cfg.ollama.default_llm_model,
    base_url=cfg.ollama.base_url,
)

available = llm_client.is_available()
models    = llm_client.list_models()

print(f"OLLAMA Server   : {'✅ Đang chạy' if available else '⚠️  Không kết nối được'}")
print(f"Base URL        : {cfg.ollama.base_url}")
print(f"LLM Model       : {cfg.ollama.default_llm_model}")
print(f"Embedding Model : {cfg.ollama.default_embedding_model}")
if models:
    print(f"Models đã pull  : {', '.join(models)}")
else:
    print("Models đã pull  : (không có hoặc server chưa chạy — xem notebook 01_ollama_setup)")

OLLAMA Server   : ⚠️  Không kết nối được
Base URL        : http://localhost:11434
LLM Model       : llama3
Embedding Model : nomic-embed-text
Models đã pull  : (không có hoặc server chưa chạy — xem notebook 01_ollama_setup)


## 2. Kiến Trúc RAGPipeline

`RAGPipeline` là **orchestrator trung tâm** — nó không chứa logic xử lý riêng, chỉ điều phối theo thứ tự qua các component được inject vào constructor (**Dependency Injection**).

```
RAGPipeline(
    loader          : BaseLoader          ← DocumentLoader
    chunker         : BaseChunker         ← TextChunker
    embedding_model : BaseEmbeddingModel  ← OllamaEmbeddingModel
    vector_store    : BaseVectorStore     ← ChromaVectorStore
    llm_client      : BaseLLMClient       ← OllamaClient
    prompt_builder  : PromptBuilder
    top_k           : int
)
```

**Ưu điểm của design này:**
- Swap component dễ dàng (ví dụ đổi ChromaVectorStore → FAISS mà không sửa RAGPipeline)
- Test từng component độc lập
- Dùng mock trong notebook khi thành phần thật chưa sẵn sàng

---

Phần còn lại của notebook đi qua từng bước, giải thích interface và correctness properties, rồi ghép lại thành pipeline hoàn chỉnh ở mục 9.

## 3. Bước 1 — Tải Tài Liệu (`DocumentLoader`)

`DocumentLoader.load(file_path)` trả về `Document` với:
- `doc_id` — hash SHA-256 của đường dẫn tuyệt đối (duy nhất, Property 12)
- `doc_type` — enum `DocumentType` (PDF/TXT/MARKDOWN/HTML)
- `content` — toàn bộ văn bản dưới dạng chuỗi

> `DocumentLoader` đã triển khai đầy đủ — S1-DE-01 (TXT/MD) và S1-DE-02 (PDF).

In [3]:
loader      = DocumentLoader()
sample_file = PROJECT_ROOT / "data" / "raw" / "sample_rag_overview.txt"

if not sample_file.exists():
    print(f"⚠️  {sample_file.name} — đang tạo file mẫu...")
    sample_file.parent.mkdir(parents=True, exist_ok=True)
    sample_file.write_text(
        "RAG (Retrieval-Augmented Generation) là phương pháp kết hợp "
        "trỳ xuất tài liệu với sinh văn bản bởi LLM cục bộ.",
        encoding="utf-8",
    )
    print(f"Đã tạo: {sample_file.name}")

doc = loader.load(str(sample_file))

print("Document đã tải:")
print(f"  doc_id   : {doc.doc_id[:20]}...")
print(f"  doc_type : {doc.doc_type.value}")
print(f"  file     : {doc.file_path}")
print(f"  length   : {len(doc.content)} ký tự")
print()
print("Nội dung đầy đủ:")
print("─" * 50)
print(doc.content)


Document đã tải:
  doc_id   : 4c69f82b6454088f4024...
  doc_type : txt
  file     : D:\lh222k\AI-Research-Assistant-with-RAG\data\raw\sample_rag_overview.txt
  length   : 1255 ký tự

Nội dung đầy đủ:
──────────────────────────────────────────────────
Retrieval-Augmented Generation (RAG) la mot kien truc ket hop giua he thong
truy xuat thong tin (retrieval) va mo hinh sinh ngon ngu (generation). Thay vi
chi dua vao kien thuc da hoc trong qua trinh huan luyen, mot he thong RAG se
tim kiem cac doan van ban lien quan tu kho du lieu rieng truoc khi yeu cau LLM
soan cau tra loi - giup cau tra loi bam sat nguon tai lieu thuc te va giam
hien tuong "ao giac" (hallucination).

Quy trinh RAG co ban gom hai giai doan chinh: indexing (tai tai lieu, chia nho
thanh chunk, tao embedding va luu vao vector store) va querying (embed cau hoi,
tim cac chunk gan nhat ve mat ngu nghia, ghep thanh prompt va goi LLM sinh cau
tra loi). Du an nay trien khai toan bo quy trinh do bang cac thanh phan chay
hoan toan

In [4]:
# Kiểm tra hành vi khi format không hỗ trợ
print("Thử load file .docx (không hỗ trợ):")
try:
    loader.load("document.docx")
except ValueError as e:
    print(f"  ValueError: {e}")

print()
print("Kiểm tra supports():")
for ext_test in [".txt", ".md", ".pdf", ".docx", ".html"]:
    fake_path = f"file{ext_test}"
    print(f"  {ext_test:8s} → supports={loader.supports(fake_path)}")

Thử load file .docx (không hỗ trợ):
  ValueError: Định dạng file .docx không được hỗ trợ

Kiểm tra supports():
  .txt     → supports=True
  .md      → supports=True
  .pdf     → supports=True
  .docx    → supports=False
  .html    → supports=False


## 4. Bước 2 — Chia Nhỏ Văn Bản (`TextChunker`)

`TextChunker.chunk(document)` trả về `List[Chunk]`, mỗi Chunk có:
- `chunk_id` — định danh duy nhất
- `doc_id` — phải bằng `document.doc_id` (**Property 2**)
- `content` — đoạn văn bản
- `start_index` / `end_index` — vị trí trong văn bản gốc

Ba chiến lược: `FIXED_SIZE` · `RECURSIVE` · `SEMANTIC`

**Correctness properties (design.md Phần 3):**
- **Property 1:** Tổng nội dung chunks bao phủ toàn bộ document gốc (không mất dữ liệu)
- **Property 2:** Mỗi `chunk.doc_id == document.doc_id`

> **Trạng thái:** `TextChunker` đang là stub (S2-DE-01/02 chưa triển khai). Cell bên dưới dùng mock minimal để minh hoạ interface. Sau Sprint 2: `from src.data.chunker import TextChunker`.

In [5]:
# ─── Mock TextChunker — thay bằng import thật sau Sprint 2 ───────────────────
class _MockTextChunker(BaseChunker):
    """
    Mock minimal cho TextChunker — fixed-size với overlap.
    Interface khớp hoàn toàn với TextChunker (design.md §2.3).
    """
    def __init__(
        self,
        strategy: ChunkStrategy = ChunkStrategy.FIXED_SIZE,
        chunk_size: int = 300,
        chunk_overlap: int = 50,
    ):
        self.strategy      = strategy
        self.chunk_size    = chunk_size
        self.chunk_overlap = chunk_overlap

    def chunk(self, document: Document):
        text    = document.content
        chunks  = []
        step    = max(1, self.chunk_size - self.chunk_overlap)
        i, idx  = 0, 0
        while i < len(text):
            end     = min(i + self.chunk_size, len(text))
            content = text[i:end]
            cid     = hashlib.md5(f"{document.doc_id}_{idx}".encode()).hexdigest()
            chunks.append(Chunk(
                chunk_id    = cid,
                doc_id      = document.doc_id,
                content     = content,
                start_index = i,
                end_index   = end,
            ))
            i   += step
            idx += 1
        return chunks
# ─────────────────────────────────────────────────────────────────────────────

chunker = _MockTextChunker(chunk_size=300, chunk_overlap=50)
chunks  = chunker.chunk(doc)

avg_len = sum(len(c.content) for c in chunks) / len(chunks)

print(f"Kết quả chunking:")
print(f"  Tổng chunks   : {len(chunks)}")
print(f"  Avg length    : {avg_len:.0f} ký tự / chunk")
print(f"  Min / Max     : {min(len(c.content) for c in chunks)} / {max(len(c.content) for c in chunks)}")
print()

# Kiểm tra correctness properties
doc_id_ok = all(c.doc_id == doc.doc_id for c in chunks)
print(f"Property 2 (doc_id khớp)      : {doc_id_ok}")
print()

print(f"{'─'*60}")
for i, c in enumerate(chunks):
    print(f"  Chunk {i:2d} | start={c.start_index:4d}, end={c.end_index:4d}, len={len(c.content):3d}")
    print(f"          {repr(c.content[:80])}")

Kết quả chunking:
  Tổng chunks   : 6
  Avg length    : 243 ký tự / chunk
  Min / Max     : 5 / 300

Property 2 (doc_id khớp)      : True

────────────────────────────────────────────────────────────
  Chunk  0 | start=   0, end= 300, len=300
          'Retrieval-Augmented Generation (RAG) la mot kien truc ket hop giua he thong\ntruy'
  Chunk  1 | start= 250, end= 550, len=300
          'van ban lien quan tu kho du lieu rieng truoc khi yeu cau LLM\nsoan cau tra loi - '
  Chunk  2 | start= 500, end= 800, len=300
          'nho\nthanh chunk, tao embedding va luu vao vector store) va querying (embed cau h'
  Chunk  3 | start= 750, end=1050, len=300
          'cuc bo thong qua OLLAMA, khong phu thuoc dich vu dam may.\n\nMoi vai tro ky su tro'
  Chunk  4 | start=1000, end=1255, len=255
          'va giao dien Streamlit; Model Engineer chiu\ntrach nhiem ket noi voi OLLAMA cho c'
  Chunk  5 | start=1250, end=1255, len=  5
          ' ke.\n'


## 5. Bước 3 — Tạo Embedding (`OllamaEmbeddingModel`)

`OllamaEmbeddingModel.embed_text(text)` gọi OLLAMA `/api/embeddings` và trả về `List[float]` có độ dài `== model.dimension`.

**Correctness properties (design.md Phần 3):**
- **Property 3:** `len(embed_text(text)) == embedding_model.dimension` với mọi text
- **Property 4:** Cùng text → cùng vector (deterministic)
- **Property 5:** `embed_batch(texts)[i] == embed_text(texts[i])` với mọi `i`

> **Trạng thái:** `OllamaEmbeddingModel` đang là stub (S2-ME-01/02 chưa triển khai). Cell bên dưới dùng mock hash-based (deterministic, không cần OLLAMA). Sau Sprint 2: `from src.embeddings.embedding_model import OllamaEmbeddingModel`.

In [6]:
# ─── Mock OllamaEmbeddingModel — thay bằng import thật sau Sprint 2 ──────────
class _MockEmbeddingModel(BaseEmbeddingModel):
    """
    Mock cho OllamaEmbeddingModel — dùng sin(hash) để tạo vector giả.
    Đảm bảo: deterministic (Property 4), dimension nhất quán (Property 3),
    embed_batch == embed_text (Property 5). Không cần OLLAMA server.
    """
    _DIM = 16  # chiều thấp để dễ quan sát trong notebook

    def embed_text(self, text: str):
        seed = int(hashlib.sha256(text.encode()).hexdigest(), 16)
        raw  = [math.sin(seed + i) for i in range(self._DIM)]
        norm = math.sqrt(sum(x**2 for x in raw)) or 1.0
        return [x / norm for x in raw]

    def embed_batch(self, texts):
        return [self.embed_text(t) for t in texts]

    @property
    def dimension(self):
        return self._DIM
# ─────────────────────────────────────────────────────────────────────────────

emb_model = _MockEmbeddingModel()

# Embed tất cả chunks
all_texts   = [c.content for c in chunks]
all_vectors = emb_model.embed_batch(all_texts)

print(f"Embedding dimension : {emb_model.dimension}")
print(f"Số vectors tạo ra   : {len(all_vectors)}")
print()

# Kiểm tra correctness properties
dim_ok  = all(len(v) == emb_model.dimension for v in all_vectors)
det_ok  = emb_model.embed_text(all_texts[0]) == emb_model.embed_text(all_texts[0])
batch_ok = all(
    all_vectors[i] == emb_model.embed_text(all_texts[i])
    for i in range(len(all_texts))
)

print(f"Property 3 (dimension nhất quán)          : {dim_ok}")
print(f"Property 4 (deterministic)                : {det_ok}")
print(f"Property 5 (embed_batch == embed_text[i]) : {batch_ok}")
print()
print("Vector của chunk 0 (16 chiều):")
print([round(x, 4) for x in all_vectors[0]])

Embedding dimension : 16
Số vectors tạo ra   : 6

Property 3 (dimension nhất quán)          : True
Property 4 (deterministic)                : True
Property 5 (embed_batch == embed_text[i]) : True

Vector của chunk 0 (16 chiều):
[-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25]


## 6. Bước 4 — Lưu vào Vector Store (`ChromaVectorStore`)

`ChromaVectorStore.add(chunks, vectors)` lưu chunks + embedding vào ChromaDB. `similarity_search(query_vector, k)` trả về `List[ScoredChunk]`.

**Correctness properties (design.md Phần 3):**
- **Property 6:** `len(results) <= k`
- **Property 7:** `results[i].score >= results[i+1].score` (giảm dần)
- **Property 8:** `0.0 <= score <= 1.0`

> **Trạng thái:** `ChromaVectorStore` đang là stub (S2-DE-03, S3-DE-01 chưa triển khai). Cell bên dưới dùng mock in-memory với cosine similarity. Sau Sprint 2-3: `from src.embeddings.vector_store import ChromaVectorStore`.

In [7]:
# ─── Mock ChromaVectorStore — thay bằng import thật sau Sprint 2-3 ───────────
class _MockVectorStore(BaseVectorStore):
    """
    Mock in-memory cho ChromaVectorStore — cosine similarity thuần Python.
    Đảm bảo Property 6, 7, 8. Không cần ChromaDB.
    """
    def __init__(self):
        self._chunks  = []
        self._vectors = []

    def add(self, chunks, vectors):
        self._chunks.extend(chunks)
        self._vectors.extend(vectors)
        return True

    def _cosine(self, a, b):
        dot = sum(x * y for x, y in zip(a, b))
        na  = math.sqrt(sum(x**2 for x in a)) or 1e-9
        nb  = math.sqrt(sum(x**2 for x in b)) or 1e-9
        return (dot / (na * nb) + 1.0) / 2.0  # map [-1,1] → [0,1]

    def similarity_search(self, query_vector, k=5):
        scored = [
            (self._cosine(query_vector, v), idx)
            for idx, v in enumerate(self._vectors)
        ]
        scored.sort(reverse=True)
        return [
            ScoredChunk(chunk=self._chunks[idx], score=score, rank=rank)
            for rank, (score, idx) in enumerate(scored[:k], 1)
        ]

    def delete_collection(self, collection_name):
        self._chunks.clear()
        self._vectors.clear()
        return True

    def get_collection_stats(self):
        return {"count": len(self._chunks)}
# ─────────────────────────────────────────────────────────────────────────────

vector_store = _MockVectorStore()
success      = vector_store.add(chunks, all_vectors)

print(f"Đã lưu vào vector store : {success}")
print(f"Collection stats         : {vector_store.get_collection_stats()}")
print()

# Demo similarity search
test_query  = "RAG là gì?"
query_vec   = emb_model.embed_text(test_query)
search_k    = 3
results     = vector_store.similarity_search(query_vec, k=search_k)

print(f"Top-{search_k} chunks cho câu hỏi: '{test_query}'")
print(f"{'─'*60}")
for r in results:
    print(f"  Rank {r.rank} | score={r.score:.4f}")
    print(f"  {repr(r.chunk.content[:100])}")
    print()

# Kiểm tra correctness properties
p6 = len(results) <= search_k
p7 = all(results[i].score >= results[i+1].score for i in range(len(results)-1))
p8 = all(0.0 <= r.score <= 1.0 for r in results)

print(f"Property 6 (len <= k)            : {p6}")
print(f"Property 7 (score giảm dần)      : {p7}")
print(f"Property 8 (score trong [0,1])   : {p8}")

Đã lưu vào vector store : True
Collection stats         : {'count': 6}

Top-3 chunks cho câu hỏi: 'RAG là gì?'
────────────────────────────────────────────────────────────
  Rank 1 | score=1.0000
  ' ke.\n'

  Rank 2 | score=1.0000
  'cuc bo thong qua OLLAMA, khong phu thuoc dich vu dam may.\n\nMoi vai tro ky su trong du an phu trach m'

  Rank 3 | score=0.0000
  'va giao dien Streamlit; Model Engineer chiu\ntrach nhiem ket noi voi OLLAMA cho ca embedding lan sinh'

Property 6 (len <= k)            : True
Property 7 (score giảm dần)      : True
Property 8 (score trong [0,1])   : True


## 7. Bước 5 — Xây Dựng Prompt (`PromptBuilder`)

`PromptBuilder.build(question, contexts)` ghép `system_prompt + context chunks + question` thành một prompt hoàn chỉnh cho LLM.

**Correctness property (design.md Phần 3):**
- **Property 9:** Prompt phải chứa nội dung `question` và tất cả `contexts`

> **Trạng thái:** `PromptBuilder` đang là stub (S3-PE-01 chưa triển khai). Cell bên dưới minh hoạ interface và cấu trúc prompt. Sau Sprint 3: `from src.generation.prompt_builder import PromptBuilder`.

In [8]:
# ─── Mock PromptBuilder — thay bằng import thật sau Sprint 3 ─────────────────
class _MockPromptBuilder:
    """
    Mock cho PromptBuilder — interface khớp hoàn toàn với design.md §2.3.
    """
    DEFAULT_SYSTEM_PROMPT = (
        "Bạn là trợ lý nghiên cứu. Hãy trả lời câu hỏi DỰA TRÊN "
        "ngữ cảnh được cung cấp. Nếu không đủ thông tin, hãy nói rõ."
    )

    def __init__(self, system_prompt: str = None):
        self.system_prompt = system_prompt or self.DEFAULT_SYSTEM_PROMPT

    def format_context(self, contexts):
        parts = []
        for i, sc in enumerate(contexts, 1):
            parts.append(f"[Đoạn {i}] (score: {sc.score:.3f})\n{sc.chunk.content}")
        return "\n\n".join(parts)

    def build(self, question: str, contexts):
        ctx_text = self.format_context(contexts)
        return (
            f"{self.system_prompt}\n\n"
            f"NGỮ CẢNH:\n{ctx_text}\n\n"
            f"CÂU HỎI: {question}\n\n"
            f"TRẢ LỜI:"
        )

    def set_system_prompt(self, new_prompt: str):
        self.system_prompt = new_prompt
# ─────────────────────────────────────────────────────────────────────────────

prompt_builder = _MockPromptBuilder()

question = "RAG hoạt động như thế nào và gồm những bước gì?"
prompt   = prompt_builder.build(question, results)

print(f"Prompt được tạo ({len(prompt)} ký tự):")
print(f"{'─'*60}")
print(prompt)
print()

# Kiểm tra Property 9
p9_question  = question in prompt
p9_contexts  = all(sc.chunk.content in prompt for sc in results)
print(f"Property 9 — prompt chứa câu hỏi       : {p9_question}")
print(f"Property 9 — prompt chứa tất cả contexts : {p9_contexts}")

Prompt được tạo (830 ký tự):
────────────────────────────────────────────────────────────
Bạn là trợ lý nghiên cứu. Hãy trả lời câu hỏi DỰA TRÊN ngữ cảnh được cung cấp. Nếu không đủ thông tin, hãy nói rõ.

NGỮ CẢNH:
[Đoạn 1] (score: 1.000)
 ke.


[Đoạn 2] (score: 1.000)
cuc bo thong qua OLLAMA, khong phu thuoc dich vu dam may.

Moi vai tro ky su trong du an phu trach mot phan rieng: Data Engineer lo viec
tai tai lieu, chia nho van ban va luu tru vector; Pipeline Engineer dieu phoi
luong RAGPipeline, xay dung prompt va giao dien Streamlit; Model Engineer chiu
trach 

[Đoạn 3] (score: 0.000)
va giao dien Streamlit; Model Engineer chiu
trach nhiem ket noi voi OLLAMA cho ca embedding lan sinh van ban. Su phoi hop
giua ba vai tro nay - dac biet o cac task phu thuoc lien vai tro - la diem mau
chot de toan bo he thong hoat dong dung nhu thiet ke.


CÂU HỎI: RAG hoạt động như thế nào và gồm những bước gì?

TRẢ LỜI:

Property 9 — prompt chứa câu hỏi       : True
Property 9 — prompt chứa tất cả 

## 8. Bước 6 — Sinh Câu Trả Lời (`OllamaClient.generate`)

`OllamaClient.generate(prompt)` gửi prompt đến OLLAMA `/api/generate` và trả về text hoàn chỉnh.

> **Trạng thái:** `OllamaClient.generate()` đang là stub (S3-ME-01 chưa triển khai). Cell bên dưới thử gọi nếu server chạy, graceful-fallback nếu `NotImplementedError`.

In [9]:
if not available:
    print("⚠️  OLLAMA server chưa chạy — bỏ qua bước sinh câu trả lời.")
    print("   Khởi động bằng: ollama serve")
else:
    print(f"Gọi {llm_client.model_name}.generate() ...")
    try:
        start   = time.time()
        answer  = llm_client.generate(prompt)
        elapsed = (time.time() - start) * 1000
        print(f"✅ Câu trả lời ({elapsed:.0f} ms):\n{'─'*50}")
        print(answer)
    except NotImplementedError:
        print("⚠️  OllamaClient.generate() chưa triển khai (S3-ME-01).")
        print("   Sẽ hoàn thiện trong Sprint 3.")
    except Exception as e:
        print(f"Lỗi khi gọi generate(): {type(e).__name__}: {e}")

⚠️  OLLAMA server chưa chạy — bỏ qua bước sinh câu trả lời.
   Khởi động bằng: ollama serve


## 9. RAGPipeline — Kết Hợp Toàn Bộ (End-to-End với Mock)

Cell bên dưới lắp ráp toàn bộ pipeline từ các mock ở trên và chạy `index_document()` + `query()` end-to-end theo đúng pseudocode design.md §2.7.

**`RAGPipeline.index_document()` (pseudocode §2.7):**
```
doc    ← loader.load(file_path)
chunks ← chunker.chunk(doc)
vectors ← [embed_text(c.content) for c in chunks]   # loop invariant: len(vectors)==i
vector_store.add(chunks, vectors)
return IndexingResult
```

**`RAGPipeline.query()` (pseudocode §2.7):**
```
query_vector ← embed_text(question)
contexts     ← similarity_search(query_vector, k=top_k)
prompt       ← prompt_builder.build(question, contexts)
answer       ← llm_client.generate(prompt)
return RAGResponse(question, answer, contexts, latency_ms)
```

In [10]:
# ─── Mock LLM Adapter — trả về câu trả lời giả khi generate() chưa sẵn sàng ─
class _MockLLMClient(BaseLLMClient):
    """Trả về câu trả lời đơn giản mà không cần OLLAMA — dùng để demo pipeline."""
    model_name = "mock-llm-v1"

    def generate(self, prompt: str, **kwargs) -> str:
        # Trích từ khóa đơn giản từ prompt để tạo câu trả lời có liên quan
        lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
        ctx_lines = [
            ln for ln in lines
            if ln.startswith("[") or (len(ln) > 30 and "CÂU HỎI" not in ln and "NGỮ CẢNH" not in ln)
        ]
        snippet = ctx_lines[0][:120] if ctx_lines else "(không có context)"
        return (
            f"[Mock Answer] Dựa trên tài liệu được cung cấp:\n"
            f"{snippet}\n"
            f"... (Câu trả lời thật sẽ được sinh bởi {cfg.ollama.default_llm_model} sau Sprint 3)"
        )

    def is_available(self) -> bool:
        return True
# ─────────────────────────────────────────────────────────────────────────────

# Khởi tạo pipeline với mock components (vector store rỗng — dùng instance mới)
fresh_store = _MockVectorStore()

pipeline = RAGPipeline(
    loader          = loader,                               # DocumentLoader thật
    chunker         = _MockTextChunker(chunk_size=300, chunk_overlap=50),
    embedding_model = _MockEmbeddingModel(),
    vector_store    = fresh_store,
    llm_client      = _MockLLMClient(),
    prompt_builder  = _MockPromptBuilder(),
    top_k           = 3,
)

# ── Giai đoạn 1: INDEXING ─────────────────────────────────────────────────────
print("=" * 55)
print("GIAI ĐOẠN 1 — INDEXING")
print("=" * 55)

index_result = pipeline.index_document(str(sample_file))

print(f"  doc_id     : {index_result.doc_id[:20]}...")
print(f"  num_chunks : {index_result.num_chunks}")
print(f"  success    : {index_result.success}")
print(f"  stats      : {fresh_store.get_collection_stats()}")

# ── Giai đoạn 2: QUERYING ────────────────────────────────────────────────────
print()
print("=" * 55)
print("GIAI ĐOẠN 2 — QUERYING")
print("=" * 55)

response = pipeline.query("RAG là gì và gồm những thành phần nào?")

print(f"  question      : {response.question}")
print(f"  model         : {response.model_name}")
print(f"  latency_ms    : {response.latency_ms:.1f} ms")
print(f"  num_contexts  : {len(response.contexts)}")
print()
print(f"  Contexts (top-3):")
for sc in response.contexts:
    print(f"    rank={sc.rank} score={sc.score:.4f} | {repr(sc.chunk.content[:80])}")
print()
print(f"  Answer:\n{'─'*55}")
print(response.answer)

GIAI ĐOẠN 1 — INDEXING
  doc_id     : 4c69f82b6454088f4024...
  num_chunks : 6
  success    : True
  stats      : {'count': 6}

GIAI ĐOẠN 2 — QUERYING
  question      : RAG là gì và gồm những thành phần nào?
  model         : mock-llm-v1
  latency_ms    : 0.0 ms
  num_contexts  : 3

  Contexts (top-3):
    rank=1 score=1.0000 | 'va giao dien Streamlit; Model Engineer chiu\ntrach nhiem ket noi voi OLLAMA cho c'
    rank=2 score=1.0000 | 'nho\nthanh chunk, tao embedding va luu vao vector store) va querying (embed cau h'
    rank=3 score=1.0000 | 'van ban lien quan tu kho du lieu rieng truoc khi yeu cau LLM\nsoan cau tra loi - '

  Answer:
───────────────────────────────────────────────────────
[Mock Answer] Dựa trên tài liệu được cung cấp:
Bạn là trợ lý nghiên cứu. Hãy trả lời câu hỏi DỰA TRÊN ngữ cảnh được cung cấp. Nếu không đủ thông tin, hãy nói rõ.
... (Câu trả lời thật sẽ được sinh bởi llama3 sau Sprint 3)


## 10. Thực Nghiệm — Index Nhiều Tài Liệu và So Sánh

Dùng `index_directory()` để index cả thư mục, rồi hỏi các câu hỏi khác nhau.

In [11]:
raw_dir = PROJECT_ROOT / "data" / "raw"

# Pipeline mới với vector store rỗng
multi_store = _MockVectorStore()
multi_pipeline = RAGPipeline(
    loader          = loader,
    chunker         = _MockTextChunker(chunk_size=300, chunk_overlap=50),
    embedding_model = _MockEmbeddingModel(),
    vector_store    = multi_store,
    llm_client      = _MockLLMClient(),
    prompt_builder  = _MockPromptBuilder(),
    top_k           = 3,
)

print("Index thư mục data/raw/ ...")
index_results = multi_pipeline.index_directory(str(raw_dir))
print(f"Đã index {len(index_results)} tài liệu:")
for r in index_results:
    status = "✅" if r.success else "❌"
    print(f"  {status} {r.doc_id[:16]}... — {r.num_chunks} chunks")
print(f"\nTổng chunks trong vector store: {multi_store.get_collection_stats()}")

# Hỏi nhiều câu hỏi
questions = [
    "RAG là gì?",
    "Vai trò của Data Engineer là gì?",
    "Pipeline Engineer phụ trách gì?",
]

print(f"\n{'═'*60}")
for q in questions:
    resp = multi_pipeline.query(q)
    top1 = resp.contexts[0] if resp.contexts else None
    print(f"\nQ: {q}")
    print(f"   Latency : {resp.latency_ms:.1f} ms")
    if top1:
        print(f"   Top-1   : score={top1.score:.4f} | {repr(top1.chunk.content[:80])}")

Bỏ qua file lỗi khi index: D:\lh222k\AI-Research-Assistant-with-RAG\data\raw\.gitkeep — Định dạng file  không được hỗ trợ


Index thư mục data/raw/ ...


Bỏ qua file lỗi khi index: D:\lh222k\AI-Research-Assistant-with-RAG\data\raw\System Design Architecture cho Roadmap Product.xmind.pdf — File không đủ nội dung để tạo chunk: D:\lh222k\AI-Research-Assistant-with-RAG\data\raw\System Design Architecture cho Roadmap Product.xmind.pdf


Bỏ qua file lỗi khi index: D:\lh222k\AI-Research-Assistant-with-RAG\data\raw\Trương Lê Huy - Numerology.pdf — File không đủ nội dung để tạo chunk: D:\lh222k\AI-Research-Assistant-with-RAG\data\raw\Trương Lê Huy - Numerology.pdf


Đã index 2 tài liệu:
  ✅ a6deecba4a341bf8... — 5 chunks
  ✅ 4c69f82b6454088f... — 6 chunks

Tổng chunks trong vector store: {'count': 11}

════════════════════════════════════════════════════════════

Q: RAG là gì?
   Latency : 0.0 ms
   Top-1   : score=1.0000 | ' ke.\n'

Q: Vai trò của Data Engineer là gì?
   Latency : 0.0 ms
   Top-1   : score=1.0000 | ' ke.\n'

Q: Pipeline Engineer phụ trách gì?
   Latency : 0.0 ms
   Top-1   : score=1.0000 | 'va giao dien Streamlit; Model Engineer chiu\ntrach nhiem ket noi voi OLLAMA cho c'


## 11. Dùng Thành Phần Thật (Template cho Sau Sprint 2-3)

Khi tất cả component đã được triển khai, chỉ cần swap import — interface và cách gọi không đổi.

| Component | Mock → Thật | Sprint |
|-----------|-------------|--------|
| `TextChunker` | `_MockTextChunker` → `from src.data.chunker import TextChunker` | S2-DE-01/02 |
| `OllamaEmbeddingModel` | `_MockEmbeddingModel` → `from src.embeddings.embedding_model import OllamaEmbeddingModel` | S2-ME-01/02 |
| `ChromaVectorStore` | `_MockVectorStore` → `from src.embeddings.vector_store import ChromaVectorStore` | S2-DE-03, S3-DE-01 |
| `PromptBuilder` | `_MockPromptBuilder` → `from src.generation.prompt_builder import PromptBuilder` | S3-PE-01 |
| `OllamaClient.generate` | `NotImplementedError` → triển khai thật | S3-ME-01 |

In [12]:
# Template — uncomment từng dòng khi component tương ứng đã sẵn sàng
# (xem bảng sprint ở mục 11)

# from src.data.chunker import TextChunker
# from src.embeddings.embedding_model import OllamaEmbeddingModel
# from src.embeddings.vector_store import ChromaVectorStore
# from src.generation.prompt_builder import PromptBuilder

# real_pipeline = RAGPipeline(
#     loader=DocumentLoader(),
#     chunker=TextChunker(
#         strategy=ChunkStrategy.RECURSIVE,
#         chunk_size=cfg.chunker.chunk_size,
#         chunk_overlap=cfg.chunker.chunk_overlap,
#     ),
#     embedding_model=OllamaEmbeddingModel(
#         model_name=cfg.ollama.default_embedding_model,
#         ollama_base_url=cfg.ollama.base_url,
#     ),
#     vector_store=ChromaVectorStore(
#         collection_name='rag_basics_demo',
#         persist_dir=None,
#     ),
#     llm_client=OllamaClient(
#         model_name=cfg.ollama.default_llm_model,
#         base_url=cfg.ollama.base_url,
#     ),
#     prompt_builder=PromptBuilder(),
#     top_k=cfg.top_k,
# )

# # Indexing
# real_result = real_pipeline.index_document(str(sample_file))
# print(f'Indexed: {real_result.num_chunks} chunks, success={real_result.success}')

# # Querying
# real_response = real_pipeline.query('RAG la gi?')
# print(f'Answer: {real_response.answer}')
# print(f'Latency: {real_response.latency_ms:.0f} ms')

print('Template san sang - uncomment sau khi Sprint 2-3 hoan thanh.')


Template san sang - uncomment sau khi Sprint 2-3 hoan thanh.


## 12. Tổng Kết

Notebook này đã đi qua toàn bộ **6 bước** của RAG pipeline:

| Bước | Component | Trạng thái | Property liên quan |
|------|-----------|-----------|-------------------|
| 1. Tải tài liệu | `DocumentLoader` | ✅ Thật (S1-DE-01/02) | Property 12 |
| 2. Chia văn bản | `TextChunker` | 🔶 Mock (S2-DE-01/02) | Property 1, 2 |
| 3. Tạo embedding | `OllamaEmbeddingModel` | 🔶 Mock (S2-ME-01/02) | Property 3, 4, 5 |
| 4. Lưu vector | `ChromaVectorStore` | 🔶 Mock (S2-DE-03, S3-DE-01) | Property 6, 7, 8 |
| 5. Xây dựng prompt | `PromptBuilder` | 🔶 Mock (S3-PE-01) | Property 9 |
| 6. Sinh câu trả lời | `OllamaClient.generate` | 🔶 Stub (S3-ME-01) | — |

**Key takeaways:**
- `RAGPipeline` dùng Dependency Injection — swap bất kỳ component nào mà không sửa orchestrator
- Mỗi component có **interface contract** rõ ràng (pre/postcondition) trong `src/interfaces.py`
- Mỗi correctness property đã được kiểm chứng trực tiếp trong notebook
- Mọi cell chạy được từ đầu đến cuối mà không gặp exception (Yêu cầu 9.4)

**Notebook tiếp theo:**
- `02_retrieval_strategies.ipynb` — so sánh dense / sparse / hybrid retrieval
- `03_prompt_engineering.ipynb` — thiết kế và tối ưu prompt template
- `04_pipeline_evaluation.ipynb` — đánh giá pipeline qua hit rate và latency